## Stage 3: Data Preprocessing & Cleaning

Removing corrupt images, preparing train/val splits
and computing class weights to handle class imbalance.

In [ ]:
import os, cv2, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

CLASS_NAMES  = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
CLASS_COLORS = ['#27ae60', '#f39c12', '#e67e22', '#e74c3c', '#8e44ad']

print("✅ Imports done")

### 3.1 Reload Dataset

In [ ]:
IMG_DIR    = Path('/content/DR_dataset/resized_train')
LABELS_CSV = Path('/content/DR_dataset/trainLabels.csv')

labels_df = pd.read_csv(LABELS_CSV)
records   = []

for _, row in tqdm(labels_df.iterrows(), total=len(labels_df), desc='Loading'):
    img_path = IMG_DIR / f"{row['image']}.jpeg"
    if img_path.exists():
        records.append({
            'filepath'  : str(img_path),
            'image'     : row['image'],
            'diagnosis' : int(row['level']),
            'class_name': CLASS_NAMES[int(row['level'])],
        })

df = pd.DataFrame(records)
print(f'✅ Images found : {len(df):,}')
print(f'   Missing      : {len(labels_df) - len(df):,}')
display(df.head())

### 3.2 Remove Corrupt Images

In [ ]:
valid, corrupt = [], []

for path in tqdm(df['filepath'], desc='Checking files'):
    img = cv2.imread(path)
    if img is None or img.size == 0:
        corrupt.append(path)
    else:
        valid.append(path)

print(f'✅ Valid   : {len(valid):,}')
print(f'❌ Corrupt : {len(corrupt)}')

df = df[df['filepath'].isin(valid)].reset_index(drop=True)
print(f'📊 Clean dataset : {len(df):,} images')

### 3.3 Image Resizing & Normalization

All images resized to 224×224 for model input.
Pixel values normalized to [0,1] range.

In [ ]:
IMG_SIZE = 224

def preprocess_image(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    return img

# Verify on one sample
sample_path = df['filepath'].iloc[0]
img         = preprocess_image(sample_path)

print(f'✅ Image shape : {img.shape}')
print(f'✅ Min value   : {img.min():.2f}')
print(f'✅ Max value   : {img.max():.2f}')

### 3.5 Train / Validation Split

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
df['fold'] = -1

for i, (_, val_idx) in enumerate(skf.split(df, df['diagnosis'])):
    df.loc[val_idx, 'fold'] = i

train_df = df[df['fold'] != 0].reset_index(drop=True)
val_df   = df[df['fold'] == 0].reset_index(drop=True)

print(f'✅ Train : {len(train_df):,}')
print(f'✅ Val   : {len(val_df):,}')

train_df.to_csv('train_split.csv', index=False)
val_df.to_csv('val_split.csv', index=False)
print('💾 Saved: train_split.csv, val_split.csv')

### 3.6 Class Weights

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(5),
    y=train_df['diagnosis'].values
)

print('⚖️  Class Weights')
print('─' * 40)
for i, (name, w) in enumerate(zip(CLASS_NAMES, class_weights)):
    print(f'  Grade {i} — {name:<22} {w:.4f}')

print(f'\n  ⚠  Grade 4 weighted {class_weights[4]/class_weights[0]:.0f}x more than Grade 0')